# M3: 유사 사례 대조와 질문·답변 근거

## tl;dr

전체 실험 완료 전에는 성능 향상을 주장하지 않습니다. 이 노트북은 고정된 설정, 실행 로그, 내부 검증 및 채택 결정을 보여 주는 실행 기록입니다. `smoke=true` 결과는 구현 점검용입니다.

## Context & Methods

A: 증상별 양성 8·음성 8의 표현과 유사도 차이, 전체 이웃 32건의 라벨 일치도를 읽습니다. B: 현재 발화와 뒤의 두 발화 묶음을 증상별 attention으로 읽습니다. 통화 전체 표현만 받는 보정 모델을 함께 학습합니다.

### Key Assumptions

기존 23,408/5,792 분할, ROBERTA-003 seed 42, 앞 256토큰, 임계값 0.5를 유지합니다. 검색 저장소는 Training만 포함하며 자기 통화·같은 통화·동일 정규화 전사본을 제외합니다. B의 답변 제거 실험은 명시적인 뒤 발화 입력을 제거하며, 전체 인코더의 문맥 정보는 남아 있습니다.

근거: [검색 결합 분류](https://arxiv.org/abs/2305.13058), [label attention](https://www.ijcai.org/proceedings/2020/461).

### 1. 실행 경로와 설정

명령은 아래 설정을 사용합니다. `RUN_TRAINING=False`는 저장된 실행 결과를 읽는 모드입니다.

In [1]:
from pathlib import Path
import json, sys, subprocess
import numpy as np
import pandas as pd
ROOT = Path.cwd()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
RESULTS = ROOT / 'outputs/m3_context_gpu/run'
TRAIN_LABEL_DIR = ROOT / 'data/raw_merged/Training/2.라벨링데이터/TL_서울_구급'
BASE_CKPT = ROOT / 'checkpoints/mission3/m3_roberta003_seed42.pt'
REFERENCE_DIR = ROOT / 'reports/mission3/ensemble/probs'
RUN_TRAINING = False
PHASE = 'pilot'
SMOKE_ROWS = 0
MAX_HOURS = 10
sys.path.insert(0, str(ROOT/'src'))
from dcc_data.mission3_context_experiment import RECIPE
print(json.dumps(RECIPE, indent=2))

{
  "head_epochs": 10,
  "head_lr": 0.001,
  "head_batch": 128,
  "weight_decay": 0.01,
  "fine_epochs": 1,
  "encoder_lr": 1e-06,
  "fine_head_lr": 0.0001,
  "fine_batch": 16,
  "micro_batch": 8,
  "bootstrap": 1000,
  "seeds": [
    42,
    43,
    44
  ]
}


## Data

기준 예측 파일 이름으로 기존 holdout을 복원하며, 세 대조군의 파일 집합과 정답 일치를 확인합니다. 공식 Validation은 최종 후보 전용 평가 명령에서만 읽습니다.

### 2. 학습 실행 및 로그

In [2]:
if RUN_TRAINING:
    command = [sys.executable, '-u', str(ROOT/'scripts/train_mission3_context.py'),
        '--train-label-dir', str(TRAIN_LABEL_DIR), '--base-ckpt', str(BASE_CKPT),
        '--reference-probs-dir', str(REFERENCE_DIR), '--out', str(RESULTS),
        '--phase', PHASE, '--smoke-rows', str(SMOKE_ROWS), '--max-hours', str(MAX_HOURS)]
    RESULTS.mkdir(parents=True, exist_ok=True)
    with (RESULTS/'execution.log').open('a', encoding='utf-8') as log:
        process = subprocess.Popen(command, cwd=ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding='utf-8')
        for line in process.stdout:
            print(line, end=''); log.write(line); log.flush()
        if process.wait(): raise RuntimeError('Training failed; inspect execution.log')
else:
    print('Saved-run inspection mode; training is not launched.')

Saved-run inspection mode; training is not launched.


## Results

### 3. 실제 실행 상태와 비교

In [3]:
summary_path = RESULTS/'metrics.json'
summary = json.loads(summary_path.read_text(encoding='utf-8')) if summary_path.exists() else {}
print('Status:', summary.get('status', 'not_run'))
print('Smoke only:', summary.get('smoke', 'unknown'))
table = pd.DataFrame([{ 'model': name, 'macro_f1': item['macro_f1'], 'micro_f1': item['micro_f1'], 'calls': item['rows']}
                      for name, item in summary.get('pilot', {}).items()])
display(table.round(4))

Status: not_run
Smoke only: unknown


""


### 4. 비교 그림

모든 막대는 0에서 시작합니다. 축소 실행의 점수는 대회 성능으로 해석할 수 없습니다.

In [4]:
import matplotlib.pyplot as plt
if not table.empty:
    fig, ax = plt.subplots(figsize=(9,4))
    bars = ax.barh(table['model'], table['macro_f1'], color='#326a91')
    ax.bar_label(bars, fmt='%.4f', padding=4)
    ax.set(xlim=(0,1), xlabel='Macro F1 (threshold 0.5)', title='Internal holdout' + (' — SMOKE ONLY' if summary.get('smoke') else ''))
    fig.tight_layout(); plt.show()
else:
    print('No measured results are available yet.')

Matplotlib is building the font cache; this may take a moment.


No measured results are available yet.


### 5. 반복·구성 요소 제거·채택 판단

paired bootstrap은 통화 단위로 같은 통화를 함께 재표집합니다. 평균 +0.003, 최소 2개 seed 개선, 95% CI 하한 > 0, Micro F1 하락 ≤ 0.005를 모두 확인합니다.

In [5]:
repeat_table = []
for method, stages in summary.get('repeat', {}).items():
    for stage, records in stages.items():
        for seed, result in zip([42,43,44], records):
            repeat_table.append(dict(method=method,stage=stage,seed=seed,macro_f1=result['macro_f1'],micro_f1=result['micro_f1']))
display(pd.DataFrame(repeat_table).round(4))
print(json.dumps(summary.get('gate', {'status':'not_available'}), ensure_ascii=False, indent=2))

""


{
  "status": "not_available"
}


## Takeaways

공식 점수 0.6540 이상일 때만 별도 패키지를 채택합니다. 목표 0.67과 실제 결과를 구분합니다. 현재 후보 교체는 이 노트북을 열거나 축소 실행하는 것으로 발생하지 않습니다.

In [6]:
print('Measured decision:', summary.get('status','not_run'))
print('Selected method:', summary.get('selected_method'))
print('Reports:', RESULTS/'report.md')
print('Official evaluation is a separate, once-only command after full Training.')

Measured decision: not_run
Selected method: None
Reports: C:\Users\sooyah\Desktop\아주대\dcc\01_prelim\wt-m3-context\outputs\m3_context_gpu\run\report.md
Official evaluation is a separate, once-only command after full Training.
